<a href="https://colab.research.google.com/github/vsujancs24-max/AI-LAB/blob/main/1WN24CS312_week8_FIRST%20ORDER%20LOGIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

def is_variable(x):
    return isinstance(x, str) and x.startswith("?")

def substitute(x, theta):
    if is_variable(x) and x in theta:
        return substitute(theta[x], theta)
    if isinstance(x, tuple):
        return tuple(substitute(i, theta) for i in x)
    return x

def unify(x, y, theta):
    if theta is None:
        return None

    x = substitute(x, theta)
    y = substitute(y, theta)

    if x == y:
        return theta

    if is_variable(x):
        theta[x] = y
        return theta

    if is_variable(y):
        theta[y] = x
        return theta

    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None
        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None
        return theta

    return None

def standardize(rule, count):
    premises, conclusion = rule
    mapping = {}

    def rename(x):
        if is_variable(x):
            if x not in mapping:
                mapping[x] = x + "_" + str(count)
            return mapping[x]
        if isinstance(x, tuple):
            return tuple(rename(i) for i in x)
        return x

    return ([rename(p) for p in premises], rename(conclusion))

def match_premises(premises, facts, theta):
    if not premises:
        yield theta
        return

    first = substitute(premises[0], theta)

    for fact in facts:
        new_theta = unify(first, fact, theta.copy())

        if new_theta is not None:
            yield from match_premises(
                premises[1:], facts, new_theta
            )

def FOL_FC_ASK(KB, rules, query):
    count = 0

    while True:
        for fact in KB:
            result = unify(fact, query, {})
            if result is not None:
                return result

        new = set()

        for rule in rules:
            count += 1
            premises, conclusion = standardize(rule, count)

            for theta in match_premises(premises, KB, {}):
                q = substitute(conclusion, theta)

                if q not in KB and q not in new:
                    new.add(q)

                    result = unify(q, query, theta.copy())

                    if result is not None:
                        return result

        if not new:
            return False

        KB.update(new)


# Knowledge Base
KB = {
    ("Parent", "John", "Mary"),
    ("Parent", "Mary", "Sam")
}

rules = [
    (
        [
            ("Parent", "?x", "?y"),
            ("Parent", "?y", "?z")
        ],
        ("Grandparent", "?x", "?z")
    )
]

query = ("Grandparent", "John", "?who")

print("Result:", FOL_FC_ASK(KB, rules, query))


Result: {'?x_1': 'John', '?y_1': 'Mary', '?z_1': 'Sam', '?who': 'Sam'}
